# 02 — Phase 2: Q measurement per checkpoint (pure measurement, no training)

Loads each saved checkpoint, runs the **frozen 512-prompt probe set** in eval
mode, and dumps `<active-run>/measurements/metrics_ckpt{N}.json` with every §5 quantity:
effective rank (+ normalized, participation ratio, top-k shares), centered &
uncentered anisotropy, dormant fraction at τ∈{0.025, 0.1}, weight norms.

Also runs the **probe-size sensitivity check** (512 vs 2048) at checkpoints
0 and 200 — spectral metrics are sample-size sensitive (Tracing paper used
~10K probes); the PI will ask for this number.

**GPU:** T4/L4 is enough (forward passes only). Log units in `compute_log.md`.

In [ ]:
%pip install -q trl==1.6.0 transformers==5.13.0 datasets==5.0.0 accelerate==1.14.0 pytest==8.4.2 numpy==2.3.5 scipy==1.16.3 pandas==2.3.3 matplotlib==3.10.6

In [ ]:
import gc, json, os, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/eaaj-pilot")
else:
    PROJECT_DIR = Path.cwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

In [ ]:
from src.repro import get_active_run

PILOT = json.loads(Path("pilot_config.json").read_text())
RUN_DIR = get_active_run(PROJECT_DIR)
CKPTS = PILOT["stage_a"]["checkpoint_steps"]
SENSITIVITY_CKPTS = [0, 200]
LAYERS = tuple(PILOT["measurement"]["layers"])
MEASURE_DIR = RUN_DIR / "measurements"
MEASURE_DIR.mkdir(exist_ok=True)
print("measuring run:", RUN_DIR)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.data import load_probe_prompts
from src.metrics import checkpoint_q_metrics

probe = load_probe_prompts()
probe_big = load_probe_prompts(big=True)
assert torch.cuda.is_available(), "Phase 2 should run on a Colab GPU"
dtype_name = PILOT["measurement"]["model_dtype"]
dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16}[dtype_name]

for n in CKPTS:
    out_path = MEASURE_DIR / f"metrics_ckpt{n}.json"
    if out_path.exists():
        print(f"ckpt {n}: already measured, skipping")
        continue
    ckpt = RUN_DIR / f"ckpt-{n}"
    if not (ckpt / "config.json").exists():
        raise FileNotFoundError(f"missing checkpoint: {ckpt}")
    tok = AutoTokenizer.from_pretrained(ckpt)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(ckpt, dtype=dtype).to("cuda")

    t0 = time.time()
    m = checkpoint_q_metrics(
        model, tok, probe, layers=LAYERS,
        batch_size=PILOT["measurement"]["batch_size"],
        max_length=PILOT["measurement"]["max_prompt_length"])
    m.update({"checkpoint": n, "run_dir": str(RUN_DIR),
              "model_dtype_requested": dtype_name,
              "wall_seconds": time.time() - t0})

    if n in SENSITIVITY_CKPTS:
        t0 = time.time()
        m_big = checkpoint_q_metrics(
            model, tok, probe_big, layers=LAYERS,
            batch_size=PILOT["measurement"]["batch_size"],
            max_length=PILOT["measurement"]["max_prompt_length"])
        m["sensitivity_2048"] = {"per_layer": m_big["per_layer"],
                                  "wall_seconds": time.time() - t0}
        for l in m["per_layer"]:
            e512, e2048 = m["per_layer"][l]["erank"], m_big["per_layer"][l]["erank"]
            print(f"  sensitivity {l}: erank 512={e512:.1f} vs 2048={e2048:.1f} "
                  f"({100*(e2048-e512)/e2048:+.1f}%)")

    out_path.write_text(json.dumps(m, indent=1))
    print(f"ckpt {n} -> {out_path}")
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(">> compute_log.md: record units + screenshot")